In [ ]:
from pathlib import Path
import os
import pandas as pd
from nautilus_trader.persistence.catalog import ParquetDataCatalog
import re
from datetime import datetime
from pathlib import Path

# For defining Instrument
import requests
import pandas as pd

from decimal import Decimal
from pathlib import Path

from nautilus_trader.model.identifiers import InstrumentId, Symbol
from nautilus_trader.model.instruments import Instrument, CryptoPerpetual
from nautilus_trader.model.objects import Currency, Price, Quantity, Money
from nautilus_trader.persistence.catalog import ParquetDataCatalog
from nautilus_trader.persistence.wranglers import TradeTickDataWrangler



In [104]:
SYMBOL = "ETHUSDT"
EXCHANGE = "BYBIT"
INSTRUMENT_ID = f"{SYMBOL}-LINEAR.{EXCHANGE}"
DATA_DIR = Path(os.environ.get("DATA_DIR", "~/desktop/tmpMarketData")).expanduser() / SYMBOL
CATALOG_DIR = Path(os.getcwd()).parent/"nautilusDataCatalog"

In [105]:
path = DATA_DIR

# 1. Compile the regex pattern once outside the loop for speed
DATE_REGEX = re.compile(r"\d{4}-\d{2}-\d{2}")

def extract_file_date(file_path: Path) -> datetime.date:
    """Extracts the date component from the filename and returns a date object

    to serve as a strict, non-brittle sorting key.
    """
    match = DATE_REGEX.search(file_path.name)
    if not match:
        # Fail immediately and loudly if a file doesn't match the required schema
        raise ValueError(
            f"Pipeline Stop: Filename '{file_path.name}' does not contain a valid YYYY-MM-DD date stamp."
        )
    
    # Convert string match to actual datetime object
    return datetime.strptime(match.group(), "%Y-%m-%d").date()

# 2. Sort the files using the explicit datetime key
raw_files = sorted(
    [f for f in path.iterdir() if f.is_file() and f.name.endswith(".csv.gz")], # ENSURES THE FILES ARE IN CHRONOLOGICAL ORDER FOR CATALOG INDEXING
    key=extract_file_date
)

assert raw_files, f"Unable to find any CSV files in directory {path}"
raw_files

[PosixPath('/Users/damensavvasavvi/desktop/tmpMarketData/ETHUSDT/ETHUSDT2020-10-21.csv.gz'),
 PosixPath('/Users/damensavvasavvi/desktop/tmpMarketData/ETHUSDT/ETHUSDT2020-10-22.csv.gz'),
 PosixPath('/Users/damensavvasavvi/desktop/tmpMarketData/ETHUSDT/ETHUSDT2020-10-23.csv.gz'),
 PosixPath('/Users/damensavvasavvi/desktop/tmpMarketData/ETHUSDT/ETHUSDT2020-10-24.csv.gz'),
 PosixPath('/Users/damensavvasavvi/desktop/tmpMarketData/ETHUSDT/ETHUSDT2020-10-25.csv.gz'),
 PosixPath('/Users/damensavvasavvi/desktop/tmpMarketData/ETHUSDT/ETHUSDT2020-10-26.csv.gz'),
 PosixPath('/Users/damensavvasavvi/desktop/tmpMarketData/ETHUSDT/ETHUSDT2020-10-27.csv.gz'),
 PosixPath('/Users/damensavvasavvi/desktop/tmpMarketData/ETHUSDT/ETHUSDT2020-10-28.csv.gz'),
 PosixPath('/Users/damensavvasavvi/desktop/tmpMarketData/ETHUSDT/ETHUSDT2020-10-29.csv.gz'),
 PosixPath('/Users/damensavvasavvi/desktop/tmpMarketData/ETHUSDT/ETHUSDT2020-10-30.csv.gz'),
 PosixPath('/Users/damensavvasavvi/desktop/tmpMarketData/ETHUSDT/ETHUS

In [106]:
# Get instrument specs from Bybit API

def get_bybit_linear_instrument_info(symbol: str, testnet: bool = False) -> dict:
    base_url = "https://api-testnet.bybit.com" if testnet else "https://api.bybit.com"

    params = {
        "category": "linear",
        "symbol": symbol.upper(),
    }

    r = requests.get(
        f"{base_url}/v5/market/instruments-info",
        params=params,
        timeout=20,
    )
    r.raise_for_status()

    payload = r.json()

    if payload["retCode"] != 0:
        raise RuntimeError(payload)

    instruments = payload["result"]["list"]

    if not instruments:
        raise ValueError(f"No Bybit linear instrument found for {symbol}")

    return instruments[0]

info = get_bybit_linear_instrument_info(SYMBOL)

In [107]:
symbol = info["symbol"]              
base_coin = info["baseCoin"]         
quote_coin = info["quoteCoin"]      
settle_coin = info["settleCoin"]     

tick_size = info["priceFilter"]["tickSize"]
qty_step = info["lotSizeFilter"]["qtyStep"]

price_precision = int(info["priceScale"])
size_precision = abs(Decimal(qty_step).as_tuple().exponent)

min_qty = info["lotSizeFilter"]["minOrderQty"]
max_qty = info["lotSizeFilter"]["maxOrderQty"]
min_notional = info["lotSizeFilter"]["minNotionalValue"]

min_price = info["priceFilter"]["minPrice"]
max_price = info["priceFilter"]["maxPrice"]

max_leverage = Decimal(info["leverageFilter"]["maxLeverage"])
margin_init = Decimal("1") / max_leverage

In [108]:
CRYPTOPERP_INSTRUMENT = CryptoPerpetual(
    instrument_id=InstrumentId.from_str(f"{symbol}-LINEAR.BYBIT"),
    raw_symbol=Symbol(symbol),

    base_currency=Currency.from_str(base_coin),
    quote_currency=Currency.from_str(quote_coin),
    settlement_currency=Currency.from_str(settle_coin),

    is_inverse=False,

    price_precision=price_precision,
    size_precision=size_precision,

    price_increment=Price.from_str(tick_size),
    size_increment=Quantity.from_str(qty_step),

    multiplier=Quantity.from_str("1"),
    lot_size=Quantity.from_str("1"),

    min_quantity=Quantity.from_str(min_qty),
    max_quantity=Quantity.from_str(max_qty),

    min_notional=Money.from_str(f"{min_notional} {quote_coin}"),
    max_notional=None,

    min_price=Price.from_str(min_price),
    max_price=Price.from_str(max_price),

    margin_init=margin_init,
    margin_maint=Decimal("0"),

    maker_fee=Decimal("0.0002"),
    taker_fee=Decimal("0.00055"),

    ts_event=0,
    ts_init=0,

    info=info,
)

CRYPTOPERP_INSTRUMENT

CryptoPerpetual(id=ETHUSDT-LINEAR.BYBIT, raw_symbol=ETHUSDT, asset_class=CRYPTOCURRENCY, instrument_class=SWAP, quote_currency=USDT, is_inverse=False, price_precision=2, price_increment=0.01, size_precision=2, size_increment=0.01, multiplier=1, lot_size=1, margin_init=0.01, margin_maint=0, maker_fee=0.0002, taker_fee=0.00055, info={'symbol': 'ETHUSDT', 'contractType': 'LinearPerpetual', 'status': 'Trading', 'baseCoin': 'ETH', 'quoteCoin': 'USDT', 'launchTime': '1615766400000', 'deliveryTime': '0', 'deliveryFeeRate': '', 'priceScale': '2', 'leverageFilter': {'minLeverage': '1', 'maxLeverage': '100.00', 'leverageStep': '0.01'}, 'priceFilter': {'minPrice': '0.01', 'maxPrice': '199999.98', 'tickSize': '0.01'}, 'lotSizeFilter': {'maxOrderQty': '10000.00', 'minOrderQty': '0.01', 'qtyStep': '0.01', 'postOnlyMaxOrderQty': '10000.00', 'maxMktOrderQty': '2000.00', 'minNotionalValue': '5'}, 'unifiedMarginTrade': True, 'fundingInterval': 480, 'settleCoin': 'USDT', 'copyTrading': 'both', 'upperFund

In [109]:
def ingest_weekly_csv(file_path: Path, instrument: Instrument):
    """
    Safely ingests a weekly csv.gz file using PyArrow into a local persistent catalog.
    """
    wrangler = TradeTickDataWrangler(instrument)
    
    # Fast, multi-threaded PyArrow parsing 
    df = pd.read_csv(
        file_path,
        engine="pyarrow",
        dtype_backend="pyarrow"
    )

    # 1. Sanity Checks
    df = df.drop_duplicates(subset=["trdMatchID"])
    df = df[(df["price"] > 0) & (df["size"] > 0)]
    df["side"] = df["side"].str.upper()

    # Rename Bybit columns to the names expected by TradeTickDataWrangler
    df = df.rename(columns={
        "size": "quantity",
        "trdMatchID": "trade_id",
    })

    # 2. The Timestamp Implementation
    # Convert ByBit Unix seconds to Unix nanoseconds
    nanoseconds = df["timestamp"] * 1_000_000_000

    # Explicitly create the two columns Nautilus demands
    df["ts_event"] = nanoseconds
    df["ts_init"] = nanoseconds 

    # Drop the old single timestamp column to avoid confusing the Wrangler
    df = df.drop(columns=["timestamp"])

    # Set the DataFrame index to a timezone-aware UTC DatetimeIndex. 
    # Nautilus requires this to exist before it will process the rows.  
    df.index = pd.to_datetime(df["ts_event"], unit="ns", utc=True)
    df.index.name = None

    # 3. Sort chronologically by the true event time
    df = df.sort_index(kind="stable")

    # Now the Wrangler has exactly what it needs for the internal structs
    return wrangler.process(df)


In [110]:
catalog = ParquetDataCatalog(str(CATALOG_DIR))
catalog.write_data([CRYPTOPERP_INSTRUMENT])

for tickDataFile in raw_files:
    catalog.write_data(ingest_weekly_csv(tickDataFile, CRYPTOPERP_INSTRUMENT))
    print(f"Successfully cataloged {tickDataFile}")

Successfully cataloged /Users/damensavvasavvi/desktop/tmpMarketData/ETHUSDT/ETHUSDT2020-10-21.csv.gz
Successfully cataloged /Users/damensavvasavvi/desktop/tmpMarketData/ETHUSDT/ETHUSDT2020-10-22.csv.gz
Successfully cataloged /Users/damensavvasavvi/desktop/tmpMarketData/ETHUSDT/ETHUSDT2020-10-23.csv.gz
Successfully cataloged /Users/damensavvasavvi/desktop/tmpMarketData/ETHUSDT/ETHUSDT2020-10-24.csv.gz
Successfully cataloged /Users/damensavvasavvi/desktop/tmpMarketData/ETHUSDT/ETHUSDT2020-10-25.csv.gz
Successfully cataloged /Users/damensavvasavvi/desktop/tmpMarketData/ETHUSDT/ETHUSDT2020-10-26.csv.gz
Successfully cataloged /Users/damensavvasavvi/desktop/tmpMarketData/ETHUSDT/ETHUSDT2020-10-27.csv.gz
Successfully cataloged /Users/damensavvasavvi/desktop/tmpMarketData/ETHUSDT/ETHUSDT2020-10-28.csv.gz
Successfully cataloged /Users/damensavvasavvi/desktop/tmpMarketData/ETHUSDT/ETHUSDT2020-10-29.csv.gz
Successfully cataloged /Users/damensavvasavvi/desktop/tmpMarketData/ETHUSDT/ETHUSDT2020-10-